# is-differentiable-flag — worked example 3: Gate 3 — requires_grad only when at least one input is tracked

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `is-differentiable-flag`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Even when `is_differentiable=True` and `grad_tracking_enabled=True`, the output requires no gradient if none of the inputs have `requires_grad=True`. This third gate prevents unnecessary Recipe allocation when the computation graph has no leaf to propagate back to.

## Worked solution

Step 1: We implement `wrap_forward_fn` with the three-gate AND. Gate 3 checks `any(isinstance(a, MiniTensor) and a.requires_grad for a in args)`.

Step 2: We test four input combinations for a differentiable op with tracking enabled:
- No MiniTensor inputs → requires_grad=False (no tracked inputs at all).
- Both inputs are MiniTensors with requires_grad=False → requires_grad=False.
- Only one input has requires_grad=True → requires_grad=True (gate 3 passes).
- Both inputs have requires_grad=True → requires_grad=True.

Step 3: We print results and confirm that gate 3 is the deciding factor in cases 1 and 2, while the per-op flag and global flag are both True throughout.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn, is_differentiable=True):
    def tensor_func(*args, **kwargs):
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        out_arr = fwd_fn(*raw_args, **kwargs)
        global grad_tracking_enabled
        gate3 = any(isinstance(a, MiniTensor) and a.requires_grad for a in args)
        requires_grad = grad_tracking_enabled and is_differentiable and gate3
        out = MiniTensor(out_arr, requires_grad)
        if requires_grad:
            parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

wrapped_mul = wrap_forward_fn(np.multiply, is_differentiable=True)

arr = np.array([2.0, 4.0])

# Case 1: no MiniTensor inputs at all (raw numpy)
out_raw = wrapped_mul(arr, arr)
print('raw inputs requires_grad:', out_raw.requires_grad)  # False

# Case 2: both MiniTensors frozen (requires_grad=False)
a_frozen = MiniTensor(arr.copy(), requires_grad=False)
b_frozen = MiniTensor(arr.copy(), requires_grad=False)
out_frozen = wrapped_mul(a_frozen, b_frozen)
print('both frozen requires_grad:', out_frozen.requires_grad)  # False
print('both frozen recipe:', out_frozen.recipe)                # None

# Case 3: one tracked, one frozen
a_tracked = MiniTensor(arr.copy(), requires_grad=True)
out_mixed = wrapped_mul(a_tracked, b_frozen)
print('one tracked requires_grad:', out_mixed.requires_grad)   # True
print('one tracked has recipe:', out_mixed.recipe is not None) # True

# Case 4: both tracked
b_tracked = MiniTensor(arr.copy(), requires_grad=True)
out_both = wrapped_mul(a_tracked, b_tracked)
print('both tracked requires_grad:', out_both.requires_grad)   # True
print('parents count:', len(out_both.recipe.parents))          # 2